# Agriculture-Vision 2017: supervised RGB U-Net baseline
Attach the miniscale data, official splits, and a code dataset containing `src/`, `configs/`, `requirements.txt`. Enable a GPU and Run All. No earlier-session variables or internet installation are required. Save run outputs before discarding the runtime.


## 1. Locate attached code and import dependencies

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import json
import shutil
import sys
from pathlib import Path

SOURCE_DIR = None  # Set explicitly only if multiple source versions are attached.
if SOURCE_DIR is None:
    candidates = sorted({p.parent.parent for p in Path("/kaggle/input").rglob("src/train.py")
                         if (p.parent.parent / "configs/baseline.json").is_file()
                         and (p.parent / "evaluate.py").is_file()})
    if len(candidates) != 1:
        raise RuntimeError(f"Attach one project code dataset or set SOURCE_DIR. Found: {candidates}")
    SOURCE_DIR = candidates[0]
SOURCE_DIR = Path(SOURCE_DIR)
CODE_DIR = Path("/kaggle/working/agriculture_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
for folder in ("src", "configs"):
    shutil.copytree(SOURCE_DIR / folder, CODE_DIR / folder, dirs_exist_ok=True)
sys.path.insert(0, str(CODE_DIR))

try:
    import torch
    import numpy as np
    import PIL
    import matplotlib
    import tqdm
except ImportError as exc:
    raise RuntimeError("Use a Kaggle PyTorch image with NumPy, Pillow, Matplotlib, and tqdm.") from exc
from IPython.display import display, Image as DisplayImage
from src.dataset import AgricultureVisionDataset, check_batch, index_splits, make_loader
from src.model import UNet
from src.train import train, validate_config
from src.evaluate import evaluate
from src.utils import seed_everything, plot_sample
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


## 2. Set paths and seed; validate official field splits
Edit these explicit settings for different Kaggle mounts or resumption.

In [ ]:
config = json.loads((CODE_DIR / "configs/baseline.json").read_text())
config["data_dir"] = "/kaggle/input/datasets/abelnathanaelh/agriculture-vision-2017/data2017_miniscale"
config["split_file"] = "/kaggle/input/datasets/abelnathanaelh/data2017-splits/data2017_splits.json"
config["output_dir"] = "/kaggle/working/agriculture_segmentation"
config["device"] = "cuda"
# config["batch_size"] = 4  # Lower before a new run if GPU memory requires it.
RESUME_FROM = None  # e.g. "/kaggle/input/my-saved-run/last.pt"

validate_config(config)
seed_everything(config["seed"])
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before training.")
paths, split_fingerprint = index_splits(config["data_dir"], config["split_file"], config["expected_counts"])
print("Official image counts:", {name: len(items) for name, items in paths.items()})
print("Split identity:", split_fingerprint)
print("GPU:", torch.cuda.get_device_name(0))


## 3. Inspect training batch shapes and synchronized target alignment
No test pixels are loaded here. Labels: Normal=0, Drydown=1, Double plant=2, Weed cluster=3, Planter skip=4, ignore=255.

In [ ]:
preview_data = AgricultureVisionDataset(config["data_dir"], paths["train"], "train",
                                        config["image_size"], config["native_size"], augment=True)
preview_loader = make_loader(preview_data, config["batch_size"], config["num_workers"],
                             torch.Generator().manual_seed(config["seed"]), cuda=True)
images, targets = next(iter(preview_loader))
check_batch(images, targets, config["image_size"])
print("RGB:", tuple(images.shape), images.dtype, float(images.min()), float(images.max()))
print("Targets:", tuple(targets.shape), targets.dtype, targets.unique().tolist())
output = Path(config["output_dir"])
output.mkdir(parents=True, exist_ok=True)
plot_sample(images[0].numpy(), targets[0].numpy(), targets[0].numpy(),
            output / "augmented_alignment.png", "Training alignment; prediction panels repeat target")
display(DisplayImage(filename=str(output / "augmented_alignment.png")))
del preview_loader, preview_data, images, targets


## 4. Initialize and check the standard U-Net
Raw five-channel logits; no pretrained weights or attention.

In [ ]:
seed_everything(config["seed"])
preview_model = UNet(config["base_channels"]).cuda()
with torch.no_grad():
    logits = preview_model(torch.zeros(1, 3, config["image_size"], config["image_size"], device="cuda"))
assert tuple(logits.shape) == (1, 5, config["image_size"], config["image_size"])
print("Logits:", tuple(logits.shape))
print("Parameters:", sum(p.numel() for p in preview_model.parameters()))
del preview_model, logits
torch.cuda.empty_cache()


## 5. Training distribution, class weights, training/validation, and checkpoints
`train` resets seed 42 and builds train/validation loaders. Before optimization it counts **training-only**, resized, unaugmented targets and prints frequencies and capped inverse-square-root weights. A resumed run restores those saved statistics and RNG state.

Weighted Cross-Entropy ignores 255. Validation mean IoU controls checkpoint selection, scheduling, and early stopping. Progress bars and per-epoch loss/mIoU/LR are shown. `last.pt` contains optimizer, scheduler, history, RNG and best model state; `best.pt` is for final evaluation. Save/download outputs to survive a discarded runtime.


In [ ]:
last_path = output / "last.pt"
# Run All resumes automatically if the same working directory was retained.
resume_path = RESUME_FROM or (str(last_path) if last_path.exists() else None)
run = train(config, resume=resume_path)
print("Best checkpoint:", run["best"])
print("Resume checkpoint:", run["last"])
distribution = json.loads((output / "class_distribution.json").read_text())
print(json.dumps(distribution, indent=2))
display(DisplayImage(filename=str(output / "learning_curves.png")))


## 6. Final official test evaluation
Only now load test pixels and evaluate the validation-selected model. Figures are cached during the same pass. Existing final results are reused on Run All. Do not tune against test results. When attaching a previously evaluated saved run, preserve its `test/` artifacts too.


In [ ]:
metrics_path = output / "test" / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print("Reusing existing final metrics; no new test inference.")
else:
    metrics = evaluate(run["best"], device=config["device"])
print("Mean IoU:", metrics["mean_iou"])
print("Anomaly-only mean IoU:", metrics["anomaly_mean_iou"])
print("Pixel accuracy:", metrics["pixel_accuracy"])
print("Classes contributing to mean IoU:", metrics["mean_iou_classes"])
for row in metrics["per_class"]:
    print(row)
print("Planter Skip:", metrics["per_class"][4])


## 7. Confusion matrix and qualitative examples
Consistent palette and gray ignore regions. Predictions use argmax. Evenly spaced samples plus a Planter Skip example if available; figures are qualitative.

In [ ]:
display(DisplayImage(filename=str(output / "test" / "confusion_matrix.png")))
manifest = json.loads((output / "test" / "visualizations.json").read_text())
for sample in manifest:
    print(sample["sample_id"], "Planter Skip:", sample["contains_planter_skip"])
    display(DisplayImage(filename=str(output / "test" / sample["figure"])))
print("Save or download run artifacts from", output)
